# Contrastive Code Entropy (CCE) - Proof of Concept

**Research Project**: Uncertainty-Guided Adaptive Context Retrieval for LLM Code Understanding

**Objective**: Validate the core hypothesis that entropy can distinguish between:
1. **Code knowledge uncertainty** (missing API/library context) → Should retrieve
2. **Language uncertainty** (word choice, phrasing) → Should NOT retrieve

**This Notebook**:
- Loads a code language model (CodeLlama or similar)
- Generates responses to code questions
- Extracts logits and computes entropy
- Tests on 10 examples (5 missing context, 5 language choice)
- Visualizes results to validate hypothesis

**Runtime**: ~20-30 minutes on Google Colab (free tier T4 GPU)

---

## Setup Instructions

1. Open in Google Colab: File → Open notebook → Upload this file
2. Enable GPU: Runtime → Change runtime type → T4 GPU
3. Run all cells: Runtime → Run all
4. Review results in final sections

---

## 1. Environment Setup

In [ ]:
%%capture
# Install required packages
!pip install transformers>=4.35.0 torch>=2.0.0 accelerate>=0.25.0 scipy numpy matplotlib seaborn pandas
!pip install -q bitsandbytes  # For 4-bit quantization (reduces memory)

In [ ]:
# Imports
import torch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from scipy.stats import entropy as scipy_entropy
from typing import List, Dict, Tuple
import warnings
warnings.filterwarnings('ignore')

# Set style for plots
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("✓ Imports successful")
print(f"✓ PyTorch version: {torch.__version__}")
print(f"✓ CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"✓ GPU: {torch.cuda.get_device_name(0)}")

## 2. Load Code Language Model

We'll use **CodeLlama-7B-Instruct** with 4-bit quantization to fit in Colab free tier (~15GB GPU RAM).

**Alternative**: If you have more GPU memory, change `load_in_4bit=False` for full precision.

In [ ]:
# Model configuration
MODEL_NAME = "codellama/CodeLlama-7b-Instruct-hf"

# 4-bit quantization config (for Colab free tier)
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

print(f"Loading model: {MODEL_NAME}")
print("This may take 2-5 minutes on first run (downloading ~3.5GB)...")

In [ ]:
# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token

# Load model with 4-bit quantization
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=quantization_config,
    device_map="auto",
    trust_remote_code=True
)

model.eval()  # Set to evaluation mode

print("✓ Model loaded successfully!")
print(f"✓ Vocabulary size: {len(tokenizer)}")
print(f"✓ Model device: {model.device}")

## 3. Entropy Calculation Functions

In [ ]:
def compute_entropy(logits: torch.Tensor) -> float:
    """
    Compute Shannon entropy from logits.
    
    H = -Σ p(x) log p(x)
    
    Args:
        logits: Tensor of shape (vocab_size,)
    
    Returns:
        Entropy value (float)
    """
    # Convert logits to probabilities
    probs = torch.softmax(logits, dim=-1)
    
    # Compute entropy (using scipy for numerical stability)
    probs_np = probs.cpu().detach().numpy()
    H = scipy_entropy(probs_np, base=2)  # bits
    
    return float(H)


def compute_normalized_entropy(logits: torch.Tensor) -> float:
    """
    Compute normalized entropy (0 to 1).
    
    H_norm = H / log2(V)
    where V = vocabulary size
    """
    H = compute_entropy(logits)
    vocab_size = logits.shape[-1]
    max_entropy = np.log2(vocab_size)
    return H / max_entropy


def compute_probability_differential(logits: torch.Tensor) -> float:
    """
    Compute probability differential (UnCert-CoT style).
    
    PD = 1 - max(P)
    """
    probs = torch.softmax(logits, dim=-1)
    max_prob = torch.max(probs).item()
    return 1.0 - max_prob


def get_top_k_tokens(logits: torch.Tensor, tokenizer, k: int = 10) -> List[Tuple[str, float]]:
    """
    Get top-k predicted tokens with probabilities.
    
    Returns:
        List of (token_string, probability) tuples
    """
    probs = torch.softmax(logits, dim=-1)
    top_k_probs, top_k_indices = torch.topk(probs, k)
    
    results = []
    for prob, idx in zip(top_k_probs, top_k_indices):
        token = tokenizer.decode([idx.item()])
        results.append((token, prob.item()))
    
    return results


print("✓ Entropy functions defined")

## 4. Token Classification (Code vs Language)

We classify tokens into:
- **Code tokens**: Programming keywords, operators, brackets
- **Language tokens**: Common English words
- **Other**: Domain-specific, variables, etc.

In [ ]:
# Programming keywords (Python, JavaScript, TypeScript, Java, Go, Rust)
CODE_KEYWORDS = {
    # Python
    'def', 'class', 'import', 'from', 'as', 'return', 'if', 'else', 'elif', 'for', 'while',
    'try', 'except', 'finally', 'with', 'lambda', 'yield', 'async', 'await', 'pass', 'break',
    'continue', 'raise', 'assert', 'None', 'True', 'False', 'and', 'or', 'not', 'in', 'is',
    
    # JavaScript/TypeScript
    'function', 'const', 'let', 'var', 'interface', 'type', 'enum', 'extends', 'implements',
    'public', 'private', 'protected', 'static', 'readonly', 'export', 'default', 'new',
    'this', 'super', 'typeof', 'instanceof', 'null', 'undefined', 'void',
    
    # Java
    'public', 'private', 'protected', 'static', 'final', 'abstract', 'synchronized', 'volatile',
    'transient', 'native', 'strictfp', 'package', 'throws', 'throw',
    
    # Go
    'func', 'package', 'import', 'var', 'const', 'type', 'struct', 'interface', 'map', 'chan',
    'go', 'defer', 'select', 'case', 'default', 'fallthrough', 'range',
    
    # Rust
    'fn', 'let', 'mut', 'const', 'static', 'struct', 'enum', 'trait', 'impl', 'pub', 'mod',
    'use', 'crate', 'match', 'loop', 'where', 'unsafe', 'async', 'await', 'move',
}

# Operators and special characters
CODE_OPERATORS = {
    '(', ')', '[', ']', '{', '}', '<', '>', '=', '==', '!=', '<=', '>=', '+', '-', '*', '/',
    '%', '&', '|', '^', '~', '<<', '>>', '&&', '||', '!', '++', '--', '+=', '-=', '*=', '/=',
    '=>', '->', '::', '.', ',', ';', ':', '?', '@', '#', '$'
}

# Common English words (subset of most frequent words)
LANGUAGE_WORDS = {
    'the', 'is', 'are', 'was', 'were', 'be', 'been', 'being', 'have', 'has', 'had', 'do',
    'does', 'did', 'will', 'would', 'should', 'could', 'may', 'might', 'must', 'can',
    'a', 'an', 'of', 'to', 'in', 'on', 'at', 'by', 'for', 'with', 'from', 'about',
    'this', 'that', 'these', 'those', 'it', 'its', 'they', 'them', 'their', 'we', 'us', 'our',
    'you', 'your', 'he', 'him', 'his', 'she', 'her', 'what', 'which', 'who', 'when', 'where',
    'why', 'how', 'all', 'each', 'every', 'both', 'few', 'more', 'most', 'other', 'some',
    'such', 'no', 'nor', 'not', 'only', 'own', 'same', 'so', 'than', 'too', 'very',
    'function', 'method', 'returns', 'takes', 'creates', 'sets', 'gets', 'handles', 'processes',
    'calculates', 'computes', 'determines', 'checks', 'validates', 'initializes', 'updates'
}


def classify_token(token: str) -> str:
    """
    Classify a token as 'code', 'language', or 'other'.
    
    Args:
        token: String token to classify
    
    Returns:
        'code', 'language', or 'other'
    """
    token_clean = token.strip().lower()
    
    # Check if code keyword or operator
    if token_clean in CODE_KEYWORDS or token in CODE_OPERATORS:
        return 'code'
    
    # Check if language word
    if token_clean in LANGUAGE_WORDS:
        return 'language'
    
    # Check for camelCase or snake_case (likely code)
    if '_' in token or (any(c.isupper() for c in token[1:]) and any(c.islower() for c in token)):
        return 'code'
    
    return 'other'


def build_token_classification(tokenizer) -> Dict[int, str]:
    """
    Build mapping from token ID to classification.
    
    Returns:
        Dict mapping token_id -> 'code'/'language'/'other'
    """
    classification = {}
    
    for token_id in range(len(tokenizer)):
        token_str = tokenizer.decode([token_id])
        classification[token_id] = classify_token(token_str)
    
    return classification


# Build token classification
print("Building token classification...")
token_classification = build_token_classification(tokenizer)

# Statistics
code_count = sum(1 for c in token_classification.values() if c == 'code')
language_count = sum(1 for c in token_classification.values() if c == 'language')
other_count = sum(1 for c in token_classification.values() if c == 'other')

print(f"✓ Token classification complete:")
print(f"  - Code tokens: {code_count} ({100*code_count/len(tokenizer):.1f}%)")
print(f"  - Language tokens: {language_count} ({100*language_count/len(tokenizer):.1f}%)")
print(f"  - Other tokens: {other_count} ({100*other_count/len(tokenizer):.1f}%)")

## 5. Contrastive Code Entropy (CCE)

In [ ]:
def compute_cce(logits: torch.Tensor, token_classification: Dict[int, str]) -> Dict:
    """
    Compute Contrastive Code Entropy (CCE).
    
    CCE = H_code - H_language
    
    Args:
        logits: Tensor of shape (vocab_size,)
        token_classification: Dict mapping token_id -> 'code'/'language'/'other'
    
    Returns:
        Dict with:
            - code_entropy: H over code tokens
            - language_entropy: H over language tokens
            - contrastive_entropy: H_code - H_language
            - total_entropy: H over all tokens
    """
    # Get probabilities
    probs = torch.softmax(logits, dim=-1)
    probs_np = probs.cpu().detach().numpy()
    
    # Separate probabilities by token type
    code_probs = []
    language_probs = []
    
    for token_id, prob in enumerate(probs_np):
        token_type = token_classification.get(token_id, 'other')
        if token_type == 'code':
            code_probs.append(prob)
        elif token_type == 'language':
            language_probs.append(prob)
    
    # Compute entropies
    total_entropy = scipy_entropy(probs_np, base=2)
    
    # Normalize and compute entropy for code tokens
    if len(code_probs) > 0:
        code_probs_norm = np.array(code_probs) / (np.sum(code_probs) + 1e-10)
        code_entropy = scipy_entropy(code_probs_norm, base=2)
    else:
        code_entropy = 0.0
    
    # Normalize and compute entropy for language tokens
    if len(language_probs) > 0:
        language_probs_norm = np.array(language_probs) / (np.sum(language_probs) + 1e-10)
        language_entropy = scipy_entropy(language_probs_norm, base=2)
    else:
        language_entropy = 0.0
    
    # Contrastive entropy
    contrastive_entropy = code_entropy - language_entropy
    
    return {
        'code_entropy': float(code_entropy),
        'language_entropy': float(language_entropy),
        'contrastive_entropy': float(contrastive_entropy),
        'total_entropy': float(total_entropy),
        'code_prob_mass': float(np.sum(code_probs)),
        'language_prob_mass': float(np.sum(language_probs))
    }


print("✓ CCE function defined")

## 6. Test Examples

We create 10 test examples:
- **5 with missing code context** (should have high code entropy)
- **5 with language choice only** (should have high language entropy)

In [ ]:
# Test examples
TEST_EXAMPLES = [
    # Group 1: Missing code context (should trigger CCE > 0)
    {
        'id': 'code_1',
        'type': 'missing_context',
        'prompt': 'How do I use the requests library to make an HTTP GET request in Python? Show me the code.',
        'context': '',  # No context about requests library
        'stop_at': 'requests.',  # Stop generation after "requests." to measure uncertainty
    },
    {
        'id': 'code_2',
        'type': 'missing_context',
        'prompt': 'Write a React component that uses useState. Show the import and component.',
        'context': '',  # No context about React
        'stop_at': 'useState(',
    },
    {
        'id': 'code_3',
        'type': 'missing_context',
        'prompt': 'How do I authenticate with Firebase in a TypeScript project? Show the auth code.',
        'context': '',  # No Firebase docs
        'stop_at': 'firebase.',
    },
    {
        'id': 'code_4',
        'type': 'missing_context',
        'prompt': 'Write a function that uses pandas to read a CSV file and filter rows.',
        'context': '',  # No pandas docs
        'stop_at': 'pd.',
    },
    {
        'id': 'code_5',
        'type': 'missing_context',
        'prompt': 'How do I create a FastAPI endpoint that handles POST requests with JSON body?',
        'context': '',  # No FastAPI docs
        'stop_at': '@app.',
    },
    
    # Group 2: Language choice only (should trigger CCE ≈ 0 or < 0)
    {
        'id': 'lang_1',
        'type': 'language_choice',
        'prompt': 'Explain what this function does: def add(a, b): return a + b',
        'context': 'def add(a, b): return a + b',  # Full context provided
        'stop_at': 'This function',  # Choosing description words
    },
    {
        'id': 'lang_2',
        'type': 'language_choice',
        'prompt': 'Write a comment describing this code: for item in items: process(item)',
        'context': 'for item in items: process(item)',
        'stop_at': '# This',  # Comment phrasing
    },
    {
        'id': 'lang_3',
        'type': 'language_choice',
        'prompt': 'Summarize this code: class User: def __init__(self, name): self.name = name',
        'context': 'class User:\n    def __init__(self, name):\n        self.name = name',
        'stop_at': 'This class',  # Descriptive text
    },
    {
        'id': 'lang_4',
        'type': 'language_choice',
        'prompt': 'Add a docstring to: def multiply(x, y): return x * y',
        'context': 'def multiply(x, y): return x * y',
        'stop_at': 'Multiplies',  # Choosing verb
    },
    {
        'id': 'lang_5',
        'type': 'language_choice',
        'prompt': 'Describe this function: def is_even(n): return n % 2 == 0',
        'context': 'def is_even(n): return n % 2 == 0',
        'stop_at': 'Returns',  # Documentation phrasing
    },
]

print(f"✓ Created {len(TEST_EXAMPLES)} test examples")
print(f"  - Missing context: {sum(1 for ex in TEST_EXAMPLES if ex['type'] == 'missing_context')}")
print(f"  - Language choice: {sum(1 for ex in TEST_EXAMPLES if ex['type'] == 'language_choice')}")

## 7. Run Experiment

For each example:
1. Generate response
2. Extract logits at key decision point
3. Compute all entropy metrics (raw, normalized, prob diff, CCE)
4. Record top-k predicted tokens

In [ ]:
def run_experiment(example: Dict) -> Dict:
    """
    Run experiment on one example.
    
    Returns:
        Dict with entropy metrics and predictions
    """
    # Format prompt
    if example['context']:
        full_prompt = f"Context:\n{example['context']}\n\nQuestion: {example['prompt']}\n\nAnswer:"
    else:
        full_prompt = f"Question: {example['prompt']}\n\nAnswer:"
    
    # Tokenize
    inputs = tokenizer(full_prompt, return_tensors="pt").to(model.device)
    
    # Generate with output_scores to get logits
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=50,
            output_scores=True,
            return_dict_in_generate=True,
            do_sample=False,  # Greedy decoding for consistency
            pad_token_id=tokenizer.eos_token_id
        )
    
    # Get generated text
    generated_ids = outputs.sequences[0][inputs['input_ids'].shape[1]:]
    generated_text = tokenizer.decode(generated_ids, skip_special_tokens=True)
    
    # Get logits from first generated token (decision point)
    # outputs.scores is a tuple of tensors, one per generated token
    first_token_logits = outputs.scores[0][0]  # Shape: (vocab_size,)
    
    # Compute all entropy metrics
    raw_entropy = compute_entropy(first_token_logits)
    normalized_entropy = compute_normalized_entropy(first_token_logits)
    prob_diff = compute_probability_differential(first_token_logits)
    cce_result = compute_cce(first_token_logits, token_classification)
    
    # Get top-k predictions
    top_k = get_top_k_tokens(first_token_logits, tokenizer, k=10)
    
    return {
        'id': example['id'],
        'type': example['type'],
        'prompt': example['prompt'],
        'generated_text': generated_text[:200],  # First 200 chars
        'raw_entropy': raw_entropy,
        'normalized_entropy': normalized_entropy,
        'probability_differential': prob_diff,
        'code_entropy': cce_result['code_entropy'],
        'language_entropy': cce_result['language_entropy'],
        'contrastive_entropy': cce_result['contrastive_entropy'],
        'total_entropy': cce_result['total_entropy'],
        'code_prob_mass': cce_result['code_prob_mass'],
        'language_prob_mass': cce_result['language_prob_mass'],
        'top_k_predictions': top_k
    }


# Run experiment on all examples
print("Running experiment on all examples...")
print("This will take ~5-10 minutes\n")

results = []
for i, example in enumerate(TEST_EXAMPLES):
    print(f"[{i+1}/{len(TEST_EXAMPLES)}] Processing {example['id']}...")
    result = run_experiment(example)
    results.append(result)
    print(f"  ✓ CCE: {result['contrastive_entropy']:.3f}, Raw H: {result['raw_entropy']:.3f}\n")

print("✓ Experiment complete!")

## 8. Results Analysis

In [ ]:
# Convert to DataFrame for analysis
df = pd.DataFrame(results)

# Summary statistics
print("=" * 80)
print("RESULTS SUMMARY")
print("=" * 80)
print("\nBy Example Type:")
print(df.groupby('type')[[
    'raw_entropy', 'normalized_entropy', 'probability_differential',
    'code_entropy', 'language_entropy', 'contrastive_entropy'
]].mean())

print("\n" + "=" * 80)
print("CCE Analysis (Key Metric)")
print("=" * 80)

missing_context_cce = df[df['type'] == 'missing_context']['contrastive_entropy'].mean()
language_choice_cce = df[df['type'] == 'language_choice']['contrastive_entropy'].mean()

print(f"\nMissing Context (should be HIGH):  CCE = {missing_context_cce:.3f}")
print(f"Language Choice (should be LOW):   CCE = {language_choice_cce:.3f}")
print(f"\nDifference: {missing_context_cce - language_choice_cce:.3f}")

if missing_context_cce > language_choice_cce:
    print("\n✓ HYPOTHESIS SUPPORTED: CCE separates the two cases!")
else:
    print("\n✗ HYPOTHESIS NOT SUPPORTED: CCE does not separate the cases.")

# Show full results table
print("\n" + "=" * 80)
print("Detailed Results")
print("=" * 80)
display(df[['id', 'type', 'raw_entropy', 'contrastive_entropy', 'code_prob_mass', 'language_prob_mass']])

## 9. Visualizations

In [ ]:
# Plot 1: Entropy comparison by type
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Raw entropy
df.groupby('type')['raw_entropy'].mean().plot(kind='bar', ax=axes[0], color=['#e74c3c', '#3498db'])
axes[0].set_title('Raw Entropy by Example Type', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Entropy (bits)', fontsize=12)
axes[0].set_xlabel('Example Type', fontsize=12)
axes[0].tick_params(axis='x', rotation=45)
axes[0].grid(axis='y', alpha=0.3)

# Contrastive Code Entropy (CCE)
df.groupby('type')['contrastive_entropy'].mean().plot(kind='bar', ax=axes[1], color=['#e74c3c', '#3498db'])
axes[1].set_title('Contrastive Code Entropy (CCE) by Example Type', fontsize=14, fontweight='bold')
axes[1].set_ylabel('CCE = H_code - H_language', fontsize=12)
axes[1].set_xlabel('Example Type', fontsize=12)
axes[1].tick_params(axis='x', rotation=45)
axes[1].axhline(y=0, color='black', linestyle='--', alpha=0.3)
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

print("Interpretation:")
print("- LEFT: Raw entropy may not distinguish the two cases well")
print("- RIGHT: CCE should be HIGHER for missing_context (code uncertainty dominant)")

In [ ]:
# Plot 2: Code vs Language Entropy Scatter
plt.figure(figsize=(10, 8))

for example_type in ['missing_context', 'language_choice']:
    subset = df[df['type'] == example_type]
    plt.scatter(
        subset['code_entropy'],
        subset['language_entropy'],
        label=example_type.replace('_', ' ').title(),
        s=150,
        alpha=0.7
    )

# Add diagonal line (code_entropy = language_entropy)
max_val = max(df['code_entropy'].max(), df['language_entropy'].max())
plt.plot([0, max_val], [0, max_val], 'k--', alpha=0.3, label='Code H = Language H')

plt.xlabel('Code Entropy (H_code)', fontsize=13)
plt.ylabel('Language Entropy (H_language)', fontsize=13)
plt.title('Code vs Language Entropy by Example Type', fontsize=15, fontweight='bold')
plt.legend(fontsize=11)
plt.grid(alpha=0.3)

# Add annotation
plt.text(
    0.05, 0.95, 
    'Points ABOVE line:\nCode uncertainty > Language uncertainty\n(CCE > 0, should retrieve)',
    transform=plt.gca().transAxes,
    bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5),
    verticalalignment='top',
    fontsize=10
)

plt.tight_layout()
plt.show()

print("\nInterpretation:")
print("- Missing context examples should be ABOVE diagonal (high code entropy)")
print("- Language choice examples should be NEAR or BELOW diagonal (low code entropy)")

In [ ]:
# Plot 3: Individual example comparison
fig, ax = plt.subplots(figsize=(14, 6))

x = np.arange(len(df))
width = 0.25

ax.bar(x - width, df['raw_entropy'], width, label='Raw Entropy', color='#95a5a6')
ax.bar(x, df['code_entropy'], width, label='Code Entropy', color='#e74c3c')
ax.bar(x + width, df['language_entropy'], width, label='Language Entropy', color='#3498db')

ax.set_xlabel('Example ID', fontsize=12)
ax.set_ylabel('Entropy (bits)', fontsize=12)
ax.set_title('Entropy Comparison Across All Examples', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(df['id'], rotation=45, ha='right')
ax.legend(fontsize=11)
ax.grid(axis='y', alpha=0.3)

# Add vertical line to separate groups
ax.axvline(x=4.5, color='black', linestyle='--', alpha=0.5)
ax.text(2, ax.get_ylim()[1] * 0.95, 'Missing Context', ha='center', fontsize=11, fontweight='bold')
ax.text(7, ax.get_ylim()[1] * 0.95, 'Language Choice', ha='center', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.show()

## 10. Inspect Top Predictions

In [ ]:
# Show top-k predictions for each example
print("=" * 80)
print("TOP-K PREDICTIONS (First Generated Token)")
print("=" * 80)

for result in results:
    print(f"\n{result['id']} ({result['type']})")
    print(f"Prompt: {result['prompt'][:80]}...")
    print(f"CCE: {result['contrastive_entropy']:.3f}")
    print("\nTop-10 predictions:")
    for i, (token, prob) in enumerate(result['top_k_predictions'], 1):
        token_type = classify_token(token)
        print(f"  {i:2d}. '{token:15s}' (p={prob:.4f}) [{token_type}]")
    print("-" * 80)

## 11. Conclusions & Next Steps

In [ ]:
# Statistical test
from scipy.stats import ttest_ind

missing_cce = df[df['type'] == 'missing_context']['contrastive_entropy']
language_cce = df[df['type'] == 'language_choice']['contrastive_entropy']

t_stat, p_value = ttest_ind(missing_cce, language_cce)

print("=" * 80)
print("FINAL ANALYSIS")
print("=" * 80)

print(f"\n1. HYPOTHESIS TEST")
print(f"   H0: CCE does not distinguish missing context from language choice")
print(f"   H1: CCE is higher for missing context examples")
print(f"\n   Results:")
print(f"   - t-statistic: {t_stat:.3f}")
print(f"   - p-value: {p_value:.4f}")

if p_value < 0.05 and missing_cce.mean() > language_cce.mean():
    print(f"   - Conclusion: ✓ REJECT H0 (p < 0.05)")
    print(f"\n   ✓ CCE successfully distinguishes the two cases!")
else:
    print(f"   - Conclusion: ✗ FAIL TO REJECT H0")
    print(f"\n   ✗ CCE does not reliably separate the cases (may need more data or refinement)")

print(f"\n2. EFFECT SIZE")
mean_diff = missing_cce.mean() - language_cce.mean()
pooled_std = np.sqrt((missing_cce.std()**2 + language_cce.std()**2) / 2)
cohens_d = mean_diff / pooled_std if pooled_std > 0 else 0
print(f"   - Mean difference: {mean_diff:.3f}")
print(f"   - Cohen's d: {cohens_d:.3f}")
print(f"   - Interpretation: ", end="")
if abs(cohens_d) < 0.2:
    print("Small effect")
elif abs(cohens_d) < 0.5:
    print("Medium effect")
else:
    print("Large effect")

print(f"\n3. NEXT STEPS")
if p_value < 0.05 and missing_cce.mean() > language_cce.mean():
    print("   ✓ Proceed to full implementation (Phase 1-7)")
    print("   - Implement CCE module in RepoSynth")
    print("   - Build adaptive retrieval system")
    print("   - Create benchmark dataset (100 examples)")
    print("   - Run experiments and write paper")
else:
    print("   ⚠ Consider adjustments before full implementation:")
    print("   - Refine token classification (use more sophisticated method)")
    print("   - Try different entropy formulations")
    print("   - Test with larger model (CodeLlama-13B)")
    print("   - Increase sample size (20-30 examples)")
    print("   - OR: Pivot to alternative approach (attention-based uncertainty)")

print("\n" + "=" * 80)

## 12. Save Results

In [ ]:
# Save results to CSV
df.to_csv('cce_poc_results.csv', index=False)
print("✓ Results saved to cce_poc_results.csv")

# Save summary
summary = {
    'missing_context_cce_mean': missing_cce.mean(),
    'language_choice_cce_mean': language_cce.mean(),
    'difference': mean_diff,
    't_statistic': t_stat,
    'p_value': p_value,
    'cohens_d': cohens_d,
    'hypothesis_supported': bool(p_value < 0.05 and missing_cce.mean() > language_cce.mean())
}

import json
with open('cce_poc_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)

print("✓ Summary saved to cce_poc_summary.json")
print("\n" + "="*80)
print("POC EXPERIMENT COMPLETE!")
print("="*80)
print("\nDownload files:")
print("- cce_poc_results.csv (detailed results)")
print("- cce_poc_summary.json (summary statistics)")
print("\nReview the visualizations and statistical tests above to make a GO/NO-GO decision.")